# Designing an A/B test: €5 vs €6 per box

**Why:** The pricing analysis (`03_pricing.ipynb`) found that €5 earns the most revenue, but it was based on what students *said*.
Before choosing a price, Dish Up should test it with real customers.

**Setup:** New app visitors are randomly split into two groups, at the same time:
- **Group A (control):** sees boxes at **€6**
- **Group B (variant):** sees boxes at **€5**

Each visitor always sees the same price. At the end, compare the two groups.

| | |
|---|---|
| **Hypothesis** | Charging €5 instead of €6 will give higher revenue per visitor, because more students will buy, enough to make up for the lower price. |
| **H0 (null)** | There is no difference in revenue per visitor between €5 and €6. |
| **Metric** | Revenue per visitor = total money ÷ visitors who saw the price. |
| **Guardrail** | Sell-out rate = % of visitors who found no boxes left. Cafés give only ~3 boxes per day, so a cheaper price could create more demand than supply. |

**Result:** To detect an increase from 10% to 15% of visitors buying, the test needs **about 681 visitors per group (~1,361 total)**,
which is **about 14 days** at 100 visitors per day.

In [1]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

## 1. Break-even point
**Assumption:** 10% of visitors buy at €6 (no real data yet, and real buying is usually much lower than what people say in surveys).

At €6: 10 out of 100 visitors buy → €60 → **€0.60 per visitor**.
How many need to buy at €5 to earn the same?

In [2]:
old_rate = 0.10
revenue_per_visitor = 6 * old_rate
break_even_rate = revenue_per_visitor / 5

print("Revenue per visitor at €6:", round(revenue_per_visitor, 2))
print("Break-even rate at €5:", round(break_even_rate, 2))

Revenue per visitor at €6: 0.6
Break-even rate at €5: 0.12


€5 needs **more than 12%** of visitors to buy. Otherwise €6 is better.

## 2. Sample size
How many visitors per group do we need before we can trust the result?
- **alpha = 0.05:** 5% chance of a false alarm (saying €5 wins when it doesn't)
- **power = 0.8:** 80% chance of catching a real difference

**Option 1:** detect the smallest useful gap, 10% → 12%.

In [3]:
effect = proportion_effectsize(0.12, 0.10)
n = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8)
print(n)

3834.595739884031


**Option 2:** detect a bigger gap, 10% → 15%.

In [4]:
effect = proportion_effectsize(0.15, 0.10)
n = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.8)
print(n)

680.3526619127882


## 3. How long would the test run?
**Assumption:** 100 visitors per day. There are **two groups**, so total visitors = n × 2.

In [5]:
visitors_per_day = 100

days_option_1 = 3835 * 2 / visitors_per_day
days_option_2 = 681 * 2 / visitors_per_day

print("10% -> 12%:", days_option_1, "days")
print("10% -> 15%:", days_option_2, "days")

10% -> 12%: 76.7 days
10% -> 15%: 13.62 days


| Gap to detect | Visitors per group | Days (at 100/day) |
|---|---|---|
| 10% → 12% | 3,835 | ~77 |
| 10% → 15% | 681 | ~14 |

A smaller gap is harder to see, so it needs many more people.

## 4. Decision and risks
**I would run the 10% → 15% test (about 14 days).** It is short enough to finish within one part of the term,
before exams or holidays change how students buy. The trade-off: if €5 is only a little better (e.g. 12%), this test may miss it.

**Risks to watch:**
- **Supply:** with ~3 boxes per café, boxes may sell out in both groups. Then we measure supply, not price. Track the sell-out rate (guardrail).
- **Fairness:** students talk to each other. Seeing different prices can feel unfair, so the test could be shown as "€1 off" instead of two prices.
- **Assumptions:** 10% buying and 100 visitors per day are guesses. Update the numbers once the app has real traffic.

**If the app has fewer visitors**, the test takes much longer. Then a simpler choice is to start at €5, keep watching revenue, and test later.